# Pre-annotation (Colab + Google Drive)

**What we are doing:** our 15,390 real sentences have no labels. Hand-labelling all of them is impossible in 3 days, so we let software draw a *first draft* of the labels, then humans only **correct** a sample.

**Why this works:** correcting a draft is ~5-10x faster than labelling from scratch, and the draft is built from three independent signals:
1. **Regex** - phone, CNIC, money, percent, time, date, law references (patterns are reliable).
2. **Gazetteers** - our lists of Pakistani cities, provinces, courts, parties, ministries, media, universities...
3. **spaCy pretrained NER** - recognises people, organisations, places in *new* sentences it never saw (this is where unseen Pakistani names are caught).

Conflicts are resolved by: strong regex first, then the longest span, then layer priority.

**Important:** the draft is NOT ground truth. It will make mistakes (e.g. a person tagged as organisation). The correction step measures exactly how often.

## 1. Mount Drive and go to the project folder
Upload the `NER-AI` folder (without `datasets/`) to Google Drive first: `My Drive/NER-AI`. Drive keeps the files after the session ends; Colab's local disk does not.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/NER-AI
!ls data/raw data/gazetteers scripts

## 2. Install spaCy and the pretrained model
`en_core_web_lg` is accurate enough for a first draft and fast on CPU. (For slightly better accuracy you can use `en_core_web_trf` with a GPU runtime.)

In [ ]:
!pip -q install spacy
!python -m spacy download en_core_web_lg

## 3. Run pre-annotation
Writes `data/annotated/preannotated.jsonl` (all sentences with draft BIO tags), plus a stratified review sample and the Label Studio config. The sample is chosen so *every* label (including rare ones) appears, rarest labels first.

In [ ]:
!python scripts/preannotate.py --model en_core_web_lg --test-n 200 --review-n 200

## 4. Look at the draft
Read a few examples yourself. This is the quickest way to learn where the draft is weak.

In [ ]:
import json, random
rows = [json.loads(l) for l in open('data/annotated/preannotated.jsonl', encoding='utf-8')]
random.seed(1)
def show(r):
    out = r['text']
    for s, e, lab, origin in sorted(r['spans'], reverse=True):
        out = out[:s] + f'[{out[s:e]} | {lab}]' + out[e:]
    return out
for r in random.sample([r for r in rows if r['spans']], 12):
    print(r['source'], '->', show(r), '\n')

## 5. Next (manual correction)
Download `data/annotated/labelstudio_tasks.json` and `labelstudio_config.xml` from Drive to your PC and review them in **Label Studio** (runs locally, see chat/README for the 4 commands). Then run `python scripts/import_labelstudio.py <export.json>` to get `corrected.jsonl` and a per-label table showing how accurate the draft was.